# MLP bằng numpy trên MNIST thật

MNIST: 70 000 ảnh chữ số viết tay 28×28 xám (60 000 train, 10 000 test). Bạn tự viết toàn bộ vòng
train của một MLP bằng numpy: đọc tệp idx gốc, forward, softmax cross-entropy, backward, SGD, rồi
tái hiện bảng SGD ở mục 6 phần lý thuyết và xem lr quá lớn giết ReLU thế nào.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, train, vẽ; test bỏ qua chúng.
  Cell đầu tiên tải MNIST vào `data/raw/mnist` nếu chưa có.
- Quy ước như phần lý thuyết và bài Basic 03: batch theo dòng, `W` shape `(in, out)`, $$\text{out} = xW + b$$.
- Chấm: `uv run pytest tracks/ai-engineer/modules/01-nn-from-scratch/exercises/intermediate/01-mlp-mnist -q`

In [ ]:
# explore: giới hạn luồng BLAS, tìm gốc repo, tải MNIST nếu chưa có (khoảng 11 MB, chỉ lần đầu)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")   # đặt TRƯỚC khi import numpy; ma trận nhỏ mà 16 luồng thì chậm hơn ~5 lần
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

MNIST = ensure("mnist")
print(MNIST, sorted(p.name for p in MNIST.iterdir()))

In [ ]:
import gzip
import time
from pathlib import Path

import numpy as np

## 1. Đọc định dạng idx

MNIST gốc là 4 tệp **idx** nén gzip. 4 byte đầu là magic number: hai byte 0, một byte mã kiểu dữ liệu
(`0x08` là `uint8`), byte thứ tư là **số chiều**. Tiếp theo, mỗi chiều một số nguyên 4 byte
**big-endian** (ảnh train: 60 000, 28, 28; nhãn: 60 000). Phần còn lại là dữ liệu `uint8` xếp liền.

`load_mnist(root)` đọc 4 tệp trong `root` (`train-images-idx3-ubyte.gz`, `train-labels-idx1-ubyte.gz`,
`t10k-images-idx3-ubyte.gz`, `t10k-labels-idx1-ubyte.gz`) và trả `(X_train, y_train, X_test, y_test)`:
ảnh làm phẳng thành 784 cột, `float32`, chia 255 về `[0, 1]`; nhãn `int64`. Không dùng torchvision.

In [ ]:
def load_mnist(root: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Đọc 4 tệp idx gz trong `root` → (X_train, y_train, X_test, y_test).

    X: float32 shape (N, 784), giá trị trong [0, 1]. y: int64 shape (N,).
    """
    def read_idx(path: Path) -> np.ndarray:
        with gzip.open(path) as f:
            data = f.read()
        ndim = data[3]                                    # magic: 0, 0, mã kiểu dữ liệu, số chiều
        shape = np.frombuffer(data, ">u4", ndim, offset=4)  # mỗi chiều 4 byte big-endian
        return np.frombuffer(data, np.uint8, offset=4 + 4 * ndim).reshape(shape)

    def images(name: str) -> np.ndarray:
        return read_idx(root / name).reshape(-1, 784).astype(np.float32) / 255

    def labels(name: str) -> np.ndarray:
        return read_idx(root / name).astype(np.int64)

    return (images("train-images-idx3-ubyte.gz"), labels("train-labels-idx1-ubyte.gz"),
            images("t10k-images-idx3-ubyte.gz"), labels("t10k-labels-idx1-ubyte.gz"))

In [ ]:
# explore: đọc MNIST thật, xem shape, phân bố nhãn và vài ảnh
import matplotlib.pyplot as plt

X_train, y_train, X_test, y_test = load_mnist(MNIST)
print(X_train.shape, X_train.dtype, X_train.min(), X_train.max(), y_train.dtype)
print("mean pixel train:", round(float(X_train.mean()), 4))
print("số ảnh mỗi lớp (train):", np.bincount(y_train).tolist())
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, label in zip(axes, X_train[:8], y_train[:8]):
    ax.imshow(img.reshape(28, 28), cmap="gray")
    ax.set_title(int(label))
    ax.axis("off")
plt.show()

## 2. MLP và một bước SGD

`MLP(sizes, seed)`: chuỗi `Linear → ReLU → ... → Linear`, tầng cuối ra **logits**.

- Khởi tạo He cho mọi `W` (phân phối chuẩn, phương sai $$2/n_{in}$$), `b` bằng 0, cả hai `float32`. Lấy số
  ngẫu nhiên lần lượt từ tầng đầu tới tầng cuối từ `np.random.default_rng(seed)`.
- `forward(X)`: trả logits, lưu cache (input từng tầng, pre-activation) cho backward.
- `loss_and_backward(X, y)`: forward, cross-entropy **trung bình trên batch** (log-softmax ổn định số),
  backward qua mọi tầng; lưu gradient, trả loss kiểu `float`.
- `params()`: danh sách cặp `(tham số, gradient)` theo thứ tự `W1, b1, W2, b2, ...`. Phải là **chính các
  mảng** trong model để optimizer sửa tại chỗ, và gradient là của lần `loss_and_backward` gần nhất.

`sgd_step(model, lr)`: $$\theta \leftarrow \theta - \eta\, g$$ cho mọi cặp, **tại chỗ**.

In [ ]:
class MLP:
    """MLP với ReLU giữa các tầng, tầng cuối ra logits (không activation).

    sizes = [784, 128, 10] nghĩa là Linear(784→128) → ReLU → Linear(128→10).
    Trọng số tầng i: W shape (sizes[i], sizes[i+1]) khởi tạo He, float32; b shape (sizes[i+1],) bằng 0, float32.
    Mọi số ngẫu nhiên lấy từ np.random.default_rng(seed).
    """

    def __init__(self, sizes: list[int], seed: int):
        rng = np.random.default_rng(seed)
        self.Ws, self.bs = [], []
        for n_in, n_out in zip(sizes[:-1], sizes[1:]):
            self.Ws.append(rng.normal(0, np.sqrt(2 / n_in), (n_in, n_out)).astype(np.float32))
            self.bs.append(np.zeros(n_out, np.float32))
        self.dWs = [np.zeros_like(W) for W in self.Ws]
        self.dbs = [np.zeros_like(b) for b in self.bs]

    def forward(self, X: np.ndarray) -> np.ndarray:
        """X (N, sizes[0]) → logits (N, sizes[-1]). Lưu cache cho backward."""
        self.inputs, self.pre = [], []                    # cache: input từng tầng, pre-activation tầng ẩn
        h = X
        last = len(self.Ws) - 1
        for i, (W, b) in enumerate(zip(self.Ws, self.bs)):
            self.inputs.append(h)
            z = h @ W + b
            if i < last:
                self.pre.append(z)
                h = np.maximum(z, 0)
            else:
                h = z
        return h

    def loss_and_backward(self, X: np.ndarray, y: np.ndarray) -> float:
        """Forward, softmax cross-entropy trung bình trên batch, backward. Lưu gradient, trả loss (float)."""
        z = self.forward(X)
        z = z - z.max(axis=1, keepdims=True)              # log-softmax ổn định số
        log_p = z - np.log(np.exp(z).sum(axis=1, keepdims=True))
        rows = np.arange(len(y))
        loss = -log_p[rows, y].mean()
        g = np.exp(log_p)                                 # dL/dlogits = (p − onehot) / N
        g[rows, y] -= 1
        g /= len(y)
        for i in reversed(range(len(self.Ws))):
            self.dWs[i] = self.inputs[i].T @ g
            self.dbs[i] = g.sum(axis=0)
            if i > 0:
                g = (g @ self.Ws[i].T) * (self.pre[i - 1] > 0)   # qua Linear rồi qua ReLU
        return float(loss)

    def params(self) -> list[tuple[np.ndarray, np.ndarray]]:
        """[(W1, dW1), (b1, db1), (W2, dW2), (b2, db2), ...]: chính các mảng của model, không phải bản sao."""
        pairs = []
        for W, b, dW, db in zip(self.Ws, self.bs, self.dWs, self.dbs):
            pairs += [(W, dW), (b, db)]
        return pairs


def sgd_step(model: MLP, lr: float) -> None:
    """p ← p − lr · grad cho mọi cặp trong model.params(), cập nhật tại chỗ."""
    for p, g in model.params():
        p -= lr * g                                       # `p = p - ...` chỉ gán lại tên cục bộ, model không đổi

## 3. Train một epoch, đo accuracy

- `train_epoch(model, X, y, lr, batch_size, rng) -> float`: xáo thứ tự bằng **một** hoán vị
  `rng.permutation(len(X))` dùng cho cả `X` và `y`, đi hết dữ liệu theo minibatch (batch cuối có thể
  ngắn hơn), mỗi batch một `loss_and_backward` + `sgd_step`. Trả loss trung bình **theo mẫu** của cả
  epoch (batch cuối ngắn thì nặng ít hơn).
- `accuracy(model, X, y) -> float`: tỉ lệ `argmax(logits) == y`.

Cùng `seed` cho model và cùng seed cho `rng` thì hai lần chạy phải ra **đúng** cùng số.

In [ ]:
def train_epoch(model: MLP, X: np.ndarray, y: np.ndarray, lr: float, batch_size: int,
                rng: np.random.Generator) -> float:
    """Một epoch SGD: xáo bằng rng, đi hết dữ liệu theo minibatch. Trả loss trung bình theo MẪU của epoch."""
    idx = rng.permutation(len(X))                         # MỘT hoán vị dùng cho cả X và y
    total = 0.0
    for start in range(0, len(X), batch_size):
        b = idx[start:start + batch_size]
        total += model.loss_and_backward(X[b], y[b]) * len(b)   # batch cuối có thể ngắn hơn
        sgd_step(model, lr)
    return total / len(X)


def accuracy(model: MLP, X: np.ndarray, y: np.ndarray) -> float:
    return float((model.forward(X).argmax(axis=1) == y).mean())

In [ ]:
# explore: kiểm trước khi train lâu (mục 7 lý thuyết): loss ở bước 0 gần ln 10, overfit được 32 ảnh
model = MLP([784, 128, 10], seed=0)
print("loss bước 0:", round(model.loss_and_backward(X_train[:1000], y_train[:1000]), 3), "| ln 10 =", round(float(np.log(10)), 3))

Xb, yb = X_train[:32], y_train[:32]
for step in range(200):
    loss = model.loss_and_backward(Xb, yb)
    sgd_step(model, lr=0.1)
print(f"overfit 32 ảnh sau 200 bước: loss {loss:.4f}, accuracy {accuracy(model, Xb, yb):.2f}")

In [ ]:
# explore: tái hiện bảng "So trên MNIST" mục 6 lý thuyết cho SGD: 784-128-10, He init, batch 64, 3 epoch, seed 0
curves = {}
for lr in [0.01, 0.1]:
    model, rng = MLP([784, 128, 10], seed=0), np.random.default_rng(0)
    start = time.perf_counter()
    curves[lr] = [train_epoch(model, X_train, y_train, lr, 64, rng) for _ in range(3)]
    print(f"SGD lr {lr:<5} train loss theo epoch {' / '.join(f'{c:.3f}' for c in curves[lr])}"
          f" | test acc {accuracy(model, X_test, y_test):.4f} | {time.perf_counter() - start:.1f} s")

for lr, curve in curves.items():
    plt.plot(range(1, 4), curve, marker="o", label=f"SGD lr {lr}")
plt.xlabel("epoch"); plt.ylabel("train loss"); plt.xticks([1, 2, 3]); plt.legend()
plt.show()

In [ ]:
# explore: thang learning rate, 1 epoch mỗi mức: lr quá lớn giết ReLU (bảng thứ hai mục 6)
def dead_fraction(model: MLP, X: np.ndarray) -> float:
    """Tỉ lệ neuron tầng ẩn đầu không bật với ảnh nào trong X."""
    (W1, _), (b1, _) = model.params()[:2]
    h = np.maximum(X @ W1 + b1, 0)
    return float((h.max(axis=0) == 0).mean())

for lr in [0.5, 1.0, 2.0, 5.0]:
    model, rng = MLP([784, 128, 10], seed=0), np.random.default_rng(0)
    idx, last = rng.permutation(len(X_train)), []
    for start in range(0, len(X_train), 64):
        b = idx[start:start + 64]
        last = (last + [model.loss_and_backward(X_train[b], y_train[b])])[-100:]   # loss 100 batch cuối
        sgd_step(model, lr)
    print(f"SGD lr {lr:<4} loss 100 batch cuối {np.mean(last):.3f} | test acc {accuracy(model, X_test, y_test):.4f}"
          f" | ReLU chết {dead_fraction(model, X_test):.0%}")

## 4. Đọc kết quả

So số của bạn với hai bảng ở mục 6 phần lý thuyết: cùng seed, cùng thứ tự lấy số ngẫu nhiên, cùng 4 luồng
BLAS thì khớp tới chữ số cuối. Máy khác hoặc số luồng khác có thể lệch ở chữ số thứ tư (float32 cộng theo
thứ tự khác); lệch từ chữ số thứ hai thường là một bug "im lặng" (mục 7): gradient không chia batch size,
xáo `X` và `y` bằng hai hoán vị khác nhau, hoặc lấy số ngẫu nhiên theo thứ tự khác.

Sau khi chạy hết notebook, làm phần **trắc nghiệm** của bài trong app.